# QLoRA SFT for Qwen3-4B-Thinking-2507

Fine-tune Qwen3-4B-Thinking on math reasoning data using QLoRA (4-bit quantization + LoRA adapters).

**Pipeline:**
1. Install / verify deps
2. Load tokenizer
3. Load 4-bit quantized base model
4. Attach LoRA adapter
5. Load + format training data (NuminaMath-CoT)
6. SFTTrainer setup
7. Train
8. Save adapter

**Evaluation** is done in the vLLM inference notebook by loading this saved adapter via `LoRARequest`.

---

**SMOKE TEST FIRST.** Set `MAX_STEPS=100` for the first run (~10 min on A30). Once that completes without OOM/error, set `MAX_STEPS=-1` for a full epoch.

In [ ]:
# Run once. Comment out after first install.
# !pip install -q --upgrade peft trl datasets bitsandbytes accelerate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import torch
from pathlib import Path

import transformers, peft, trl, datasets, bitsandbytes
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("datasets:", datasets.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

assert torch.cuda.is_available(), "No GPU detected."
print("GPU:", torch.cuda.get_device_name(0))

## Configuration

All hyperparameters in one place. Key knobs:
- `MAX_STEPS`: `100` for smoke, `-1` for full epoch
- `MAX_TRAIN_EXAMPLES`: `500` for smoke, `5000+` for real
- `MAX_SEQ_LEN`: lower this if you OOM

In [ ]:
# ── Run identity ─────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
RUN_NAME    = "qlora_sft_numinamath_v1"
OUTPUT_DIR  = f"outputs/{RUN_NAME}"
ADAPTER_OUT = f"{OUTPUT_DIR}/final_adapter"

# ── Training data ────────────────────────────────────────────────
TRAIN_DATASET      = "AI-MO/NuminaMath-CoT"
MAX_TRAIN_EXAMPLES = 500       # SMOKE: 500. REAL: 5000-20000.
MAX_SEQ_LEN        = 4096      # lower (e.g. 2048) if OOM

# ── LoRA ─────────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Optimization ─────────────────────────────────────────────────
BATCH_SIZE       = 1            # per-device. Keep small with QLoRA.
GRAD_ACCUM       = 8            # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 1
MAX_STEPS        = 100          # SMOKE: 100. REAL: -1 (full epoch).
WARMUP_RATIO     = 0.03
LOG_STEPS        = 10
SAVE_STEPS       = 200
SEED             = 42

print(f"RUN_NAME      : {RUN_NAME}")
print(f"OUTPUT_DIR    : {OUTPUT_DIR}")
print(f"MAX_STEPS     : {MAX_STEPS}  ({'SMOKE TEST' if MAX_STEPS > 0 else 'FULL EPOCH'})")
print(f"Train examples: {MAX_TRAIN_EXAMPLES}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

## Load tokenizer

In [ ]:
from transformers import AutoTokenizer
from transformers.models.qwen2.tokenization_qwen2 import Qwen2Tokenizer

# Same Qwen tokenizer patch as the vLLM notebook
if not hasattr(Qwen2Tokenizer, "all_special_tokens_extended"):
    @property
    def _all_special_tokens_extended(self):
        return list(self.all_special_tokens)
    Qwen2Tokenizer.all_special_tokens_extended = _all_special_tokens_extended

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"     # right-pad for SFT (causal LM)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Pad token:", tokenizer.pad_token, "(id:", tokenizer.pad_token_id, ")")

## Load base model (4-bit) + prepare for k-bit training

Using NF4 quantization with bf16 compute. Roughly fits in ~6GB so leaves room for activations on a 24GB A30.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True
)
model.config.use_cache = False         # required when using gradient checkpointing

print("Base model loaded. Memory:")
print(f"  allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"  reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB")

## Attach LoRA adapter

Targeting all attention + MLP projections. Trainable params should be roughly 0.5–1% of total.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Load + format training data

NuminaMath-CoT is ~860k math problems with chain-of-thought solutions. We sample a subset for the smoke test.

Each example is formatted as a Qwen chat template `system → user (problem) → assistant (solution)`. SFTTrainer will compute loss only on the assistant turn (default behavior with `assistant_only_loss=False` is loss on full sequence; we keep it simple here).

In [ ]:
from datasets import load_dataset

print(f"Loading {TRAIN_DATASET}...")
raw_train = load_dataset(TRAIN_DATASET, split="train")
print(f"Loaded {len(raw_train)} examples. Columns: {raw_train.column_names}")
print("\nSample:")
print(json.dumps({k: str(v)[:300] for k, v in raw_train[0].items()}, indent=2))

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful math solver. "
    "Solve the problem step by step, keeping the reasoning concise. "
    "Put the final answer in \\boxed{}."
)

def format_example(example):
    """NuminaMath-CoT has 'problem' and 'solution' fields."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["problem"]},
        {"role": "assistant", "content": example["solution"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

# Subsample
if 0 < MAX_TRAIN_EXAMPLES < len(raw_train):
    raw_train = raw_train.shuffle(seed=SEED).select(range(MAX_TRAIN_EXAMPLES))

train_dataset = raw_train.map(format_example, remove_columns=raw_train.column_names)
print(f"Formatted {len(train_dataset)} examples.")
print("\n── Sample formatted text (first 1500 chars) ──")
print(train_dataset[0]["text"][:1500])
print("...")
print("\n── Sample formatted text (last 500 chars) ──")
print(train_dataset[0]["text"][-500:])

## SFTTrainer setup

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field="text",
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Trainer ready.")
print(f"  Total optimization steps (this run): {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'computed at train()'}")

## Train

For the smoke run, watch:
- Loss should drop within the first 20-30 steps
- GPU memory should plateau (no slow leak)
- No NaN/inf in loss

In [ ]:
import time
t0 = time.time()
trainer.train()
print(f"\nTraining complete in {(time.time()-t0)/60:.1f} min.")

## Save adapter

In [ ]:
Path(ADAPTER_OUT).mkdir(parents=True, exist_ok=True)
trainer.save_model(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f"Adapter saved to {ADAPTER_OUT}")
print("\nFiles:")
for p in sorted(Path(ADAPTER_OUT).iterdir()):
    print(f"  {p.name:40s}  {p.stat().st_size/1e6:.2f} MB")

## How to evaluate this adapter

In your **vLLM inference notebook**, modify the model loading cell:

```python
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

vllm_model = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.75,
    max_model_len=32768,
    max_num_seqs=8,
    enable_prefix_caching=True,
    enable_lora=True,                # NEW: enable LoRA
    max_lora_rank=16,                # NEW: must match LORA_R
)

LORA_REQUEST = LoRARequest(
    lora_name="qlora_sft_v1",
    lora_int_id=1,
    lora_path="outputs/qlora_sft_numinamath_v1/final_adapter",
)
```

Then in the generate call:

```python
outputs = vllm_model.generate(
    prompts,
    sampling_params=sampling_params_sc,
    lora_request=LORA_REQUEST,        # NEW
)
```

Run the same evaluation and compare to the base-model number from your prompt-v2 row. **If the SFT-tuned model does not beat the base model by at least 3 points on the full public set, the adapter is not worth submitting.**

## Notes / things to try if smoke test passes

- **More data**: bump `MAX_TRAIN_EXAMPLES` to 5000-20000 and remove `MAX_STEPS` cap
- **Different dataset**: `AI-MO/NuminaMath-1.5`, `lighteval/MATH`, or competition-style datasets
- **Longer context**: raise `MAX_SEQ_LEN` to 8192 if you have headroom (will slow training)
- **GRPO** (advanced): if SFT works, the next-level move is RL with a math correctness reward. `trl.GRPOTrainer` supports this directly. Qwen3-Thinking was already RL-trained, so this is the most promising path.
- **Distillation from larger Qwen**: NOT allowed by competition rules (no external model calls). Stick to the base 4B.